# 08 — Phân tích lỗi VideoMAE V2 + RGB Transformer baseline top-50

Notebook này **không train**. Nó đọc prediction/artifact do notebook 07 lưu trên Drive, tạo confusion matrix, precision/recall/F1 theo từ, biểu đồ confidence và các hướng nhầm phổ biến. Mặc định chỉ phân tích `validation` để từ đó đề xuất nhánh pose; không dùng test để chỉnh mô hình.

```text
validation predictions ─► confusion / recall / confidence ─► giả thuyết lỗi RGB
                                                            └► thiết kế cải tiến pose + fusion
```

In [ ]:
from pathlib import Path
from google.colab import drive

PROJECT_GIT_URL = 'https://github.com/stillthethrone/silent-signal.git'
PROJECT_GIT_REF = 'feat/asl-citizen-videomaev2-demo-baseline'  # @param {type:'string'}
RESULTS_ROOT_STR = '/content/drive/.shortcut-targets-by-id/1-oYEcvJh4ylv_f4AKkJkCjs3FgzjDBFE/silent-signal-results/asl_citizen'  # @param {type:'string'}
BASELINE_RUN_NAME = 'videomaev2_rgb_transformer_demo50_e50_regularized_v1'  # @param {type:'string'}
ANALYSIS_SPLIT = 'validation'  # @param ['validation', 'test']
ALLOW_TEST_ANALYSIS = False  # @param {type:'boolean'}
TOP_ERRORS = 15  # @param {type:'integer'}
RUN_ANALYSIS = True  # @param {type:'boolean'}

DRIVE_MOUNT = Path('/content/drive')
def drive_ready():
    return (DRIVE_MOUNT / 'MyDrive').exists() or (DRIVE_MOUNT / '.shortcut-targets-by-id').exists()

if not drive_ready():
    try:
        drive.mount(str(DRIVE_MOUNT), force_remount=False, timeout_ms=120_000)
    except ValueError as first_error:
        print('Drive mount lần đầu thất bại; thử force_remount một lần...', flush=True)
        try:
            drive.mount(str(DRIVE_MOUNT), force_remount=True, timeout_ms=120_000)
        except ValueError as retry_error:
            raise RuntimeError(
                'Không mount được Google Drive sau 2 lần. Chọn Runtime > Disconnect and delete runtime, '
                'kết nối lại bằng đúng tài khoản có shortcut silent-signal-results rồi chạy lại cell này.'
            ) from retry_error
if not drive_ready():
    raise RuntimeError('Drive mount không báo lỗi nhưng không thấy MyDrive hoặc shortcut targets.')
print('Drive READY:', DRIVE_MOUNT, flush=True)

if ANALYSIS_SPLIT == 'test' and not ALLOW_TEST_ANALYSIS:
    raise RuntimeError('Không phân tích test để chỉnh mô hình. Chỉ bật sau khi đã chốt thiết kế bằng validation.')
PROJECT_ROOT = Path('/content/silent-signal')
RESULTS_ROOT = Path(RESULTS_ROOT_STR)
BASELINE_ROOT = RESULTS_ROOT / 'subsets/asl_citizen_asllex_top200/baselines' / BASELINE_RUN_NAME

## Môi trường CPU và kiểm tra artifact đầu vào

In [ ]:
import os, subprocess, sys, time
def run(command, *, env=None):
    command = list(map(str, command))
    started = time.perf_counter(); print('+', ' '.join(command), flush=True)
    subprocess.run(command, check=True, env=env)
    print(f'DONE {time.perf_counter() - started:.1f}s', flush=True)
if not PROJECT_ROOT.exists():
    run(['git', 'clone', '--branch', PROJECT_GIT_REF, '--single-branch', PROJECT_GIT_URL, PROJECT_ROOT])
else:
    run(['git', '-C', PROJECT_ROOT, 'fetch', 'origin', PROJECT_GIT_REF])
    run(['git', '-C', PROJECT_ROOT, 'checkout', PROJECT_GIT_REF])
    run(['git', '-C', PROJECT_ROOT, 'pull', '--ff-only', 'origin', PROJECT_GIT_REF])
ANALYSIS_SITE = Path('/content/silent-signal-analysis-site-py313-v1')
ANALYSIS_SITE.mkdir(parents=True, exist_ok=True)
analysis_packages = ['numpy==2.2.2', 'scipy==1.15.1', 'pandas==2.2.3', 'scikit-learn==1.6.1', 'seaborn==0.13.2', 'matplotlib==3.10.0']
import_check = 'import numpy, scipy, pandas, sklearn, seaborn, matplotlib; print("Environment PASS | numpy", numpy.__version__, "| scipy", scipy.__version__, "| pandas", pandas.__version__, "| sklearn", sklearn.__version__)'
ANALYSIS_ENV = dict(os.environ)
ANALYSIS_ENV['PYTHONPATH'] = os.pathsep.join([str(ANALYSIS_SITE), str(PROJECT_ROOT / 'src')])
probe = subprocess.run([sys.executable, '-c', import_check], text=True, capture_output=True, env=ANALYSIS_ENV)
if probe.returncode != 0:
    print('Isolated analysis stack missing/incompatible; installing wheels into', ANALYSIS_SITE, flush=True)
    run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-cache-dir', '--upgrade', '--target', ANALYSIS_SITE, *analysis_packages])
run([sys.executable, '-c', import_check], env=ANALYSIS_ENV)
required = [BASELINE_ROOT / 'selected_50_words.json', BASELINE_ROOT / 'manifests/train.csv', BASELINE_ROOT / 'manifests/validation.csv', BASELINE_ROOT / 'manifests/test.csv', BASELINE_ROOT / 'baseline_report.json', BASELINE_ROOT / f'{ANALYSIS_SPLIT}_predictions.csv']
missing = [str(path) for path in required if not path.is_file()]
if missing: raise FileNotFoundError('Notebook 07 chưa tạo đủ artifact: ' + ', '.join(missing))
print('Input PASS:', *required, sep='\n- ')

## Chạy phân tích với log

Nếu notebook 07 giới hạn `MAX_EVAL_BATCHES`, report sẽ đánh dấu `partial_evaluation=true`; biểu đồ chỉ mô tả phần đã đánh giá, không được gọi là kết quả cuối.

In [ ]:
command = [sys.executable, '-u', '-m', 'silent_signal.cli.analyze_videomaev2_demo',
    '--baseline-root', BASELINE_ROOT, '--split', ANALYSIS_SPLIT, '--top-errors', TOP_ERRORS]
if RUN_ANALYSIS: run(command, env=ANALYSIS_ENV)
else: print('RUN_ANALYSIS=False — chưa tạo báo cáo lỗi.')

## Dashboard lỗi baseline

In [ ]:
import json
from IPython.display import Image, display
report_path = BASELINE_ROOT / f'{ANALYSIS_SPLIT}_error_analysis.json'
if not report_path.is_file(): raise FileNotFoundError(report_path)
report = json.loads(report_path.read_text(encoding='utf-8'))
summary_keys = ('study_stage', 'analysis_split', 'evaluated_samples', 'available_samples', 'partial_evaluation', 'top1_accuracy', 'macro_f1', 'generalization', 'class_support', 'high_confidence_errors_at_0_8')
print(json.dumps({key: report[key] for key in summary_keys}, ensure_ascii=False, indent=2))
for name in ('training_curves.png', 'selected_50_official_split_counts.png', f'{ANALYSIS_SPLIT}_confusion_matrices.png', f'{ANALYSIS_SPLIT}_per_class_metrics.png', f'{ANALYSIS_SPLIT}_top_confusions.png', f'{ANALYSIS_SPLIT}_confidence_histogram.png'):
    path = BASELINE_ROOT / name
    if path.is_file():
        print('\n', name); display(Image(filename=str(path)))

## Cách dùng kết quả cho bước nghiên cứu tiếp theo

- Nhóm từ RGB dễ nhầm nhưng khác rõ về hình học tay/cơ thể là ứng viên để kiểm chứng nhánh pose.
- Confidence cao nhưng dự đoán sai gợi ý shortcut theo nền, người ký hoặc appearance.
- `generalization.best_checkpoint.top1_generalization_gap` cho biết overfit ngay tại checkpoint được chọn; không dùng metric epoch cuối để đại diện model.
- `class_support.analysis_classes_below_5` cảnh báo validation quá ít mẫu ở một số lớp. Không tự chuyển clip giữa các official split để làm metric đẹp hơn.
- Recall thấp chỉ là tín hiệu tạo giả thuyết; cần kiểm tra support và chạy lại full 200 lớp.
- Sau khi chốt kiến trúc bằng validation mới chạy test một lần để báo cáo cuối.